# Simulação do acoplamento hidráulico-térmico
### Estudo da influência do sistema hidráulico na condutividade térmica da placa, e a influência da placa térmica na viscosidade hidráulica.
#### Disciplina: SME0602 — Motores Numéricos para Simulação em Engenharia
#### Professor: Roberto F. Ausas
#### Grupo 3: 
* #### Beatriz Cosimatti
* #### Cecilia Queiroz
* #### Gabriel Zago
* #### Matheus Buzzon
* #### Pedro Vale
* #### Victor Silva


In [ ]:
# Importações!!!

import sys
import os
import importlib
from IPython.display import Video

root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if root not in sys.path:
    sys.path.insert(0, root)

import hydraulic_thermal
import themal_hydraulics
importlib.reload(hydraulic_thermal)
importlib.reload(themal_hydraulics)
import env
importlib.reload(env)
config = env.CONFIG_HT

##### Dedução da Regra do Trapézio Composta
 
##### 1.1 Hipóteses e Discretização
 
Deseja-se calcular a integral de uma grandeza $T_k(p(s))$ ao longo de uma aresta (microcanal) $k$ de comprimento $\ell_k$.
 
$$\int_0^{\ell_k} T_k(p(s))\, ds \approx \sum_{n=1}^{N} T_k(s_n)\, \Delta s, \qquad s_n = \left(n - \frac{1}{2}\right)\Delta s \tag{1}$$
 
Para a formulação via **Regra do Trapézio Composta**, a hipótese principal é aproximar a função $T_k(s)$ por segmentos de reta (funções lineares) entre pontos nodais adjacentes, em vez de assumir valor constante no centro do subintervalo.
 
Divide-se o intervalo $[0, \ell_k]$ em $N$ subintervalos uniformes de comprimento $\Delta s = \ell_k / N$. Os pontos de quadratura são definidos nos extremos de cada subintervalo:
 
$$s_n = n\,\Delta s, \qquad n = 0, 1, \ldots, N \tag{2}$$
 
##### 1.2 Dedução
 
Em cada subintervalo $[s_{n-1}, s_n]$, a integral corresponde à área de um trapézio:
 
$$\int_{s_{n-1}}^{s_n} T_k(s)\, ds \approx \frac{T_k(s_{n-1}) + T_k(s_n)}{2}\,\Delta s \tag{3}$$
 
Somando as contribuições sobre todos os $N$ subintervalos, obtemos a integral total sobre a aresta:
 
$$\int_0^{\ell_k} T_k(p(s))\, ds \approx \Delta s \left[ \frac{T_k(s_0)}{2} + \sum_{n=1}^{N-1} T_k(s_n) + \frac{T_k(s_N)}{2} \right] \tag{4}$$
 
Esta expressão pode ser simplificada introduzindo a notação $\sum''$, que indica que os termos avaliados nos extremos ($n = 0$ e $n = N$) são ponderados pelo peso $1/2$:
 
$$\int_0^{\ell_k} T_k(p(s))\, ds \approx \frac{\Delta s}{2} \sum_{n=0}^{N}{}'' T_k(s_n) \tag{5}$$
 
Ou, equivalentemente:
 
$$\int_0^{\ell_k} T_k(p(s))\, ds \approx \frac{\ell_k}{N} \left[ \frac{T_k(0) + T_k(\ell_k)}{2} + \sum_{n=1}^{N-1} T_k(s_n) \right] \tag{6}$$
 


##### Para estudar a influência da parte térmica no sistema hidráulico, implementou-se um interpolador bidimensional para estimar a temperatura em coordenadas de interesse. Comparou-se o que acontece ao usar os diferentes métodos de interpolação: nearest, cubic e linear, e ao usar diferentes discretizações iniciais: (241, 121) ou (61, 31).

In [ ]:
thermal_to_hydraulic = hydraulic_thermal.HydraulicThermal(config)
thermal_to_hydraulic.run_interpolator(plot=True)

##### Com o interpolador montado, nós calculamos a temperatura média em cada aresta da rede usando as regras de quadratura

In [ ]:
thermal_to_hydraulic.evaluate_coupling(plot=True)

##### Sabendo a temperatura de cada cano, foi possível resolver o circuito hidráulico da rede, atualizando a condutância de cada aresta para levar a temperatura média em conta na hora de calcular a viscosidade

In [ ]:
thermal_to_hydraulic.avaliar_impacto_malha_e_quadratura()

##### Cálculo de uma Viscosidade Média nas Arestas
 
##### O Problema da Abordagem Clássica
 
A abordagem tradicionalmente adotada consiste em calcular a temperatura média $\langle T_k \rangle$ da aresta e avaliar a viscosidade uma única vez nesse valor médio: $\mu(\langle T_k \rangle)$.
 
No entanto, a viscosidade dinâmica da água é descrita por uma relação **não linear** com a temperatura:
 
$$\mu(T) = \frac{0.001791}{1 + 0.03368\,T + 0.000221\,T^2} \tag{7}$$
 
##### Abordagem Proposta: Integração Direta
 
Em vez de avaliar a média da temperatura e aplicar a lei constitutiva, propõe-se calcular a viscosidade local em cada ponto de discretização e então obter o seu valor médio integrado na aresta:
 
$$\langle \mu_k \rangle = \frac{1}{\ell_k} \int_0^{\ell_k} \mu\!\left(T_k(p(s))\right) ds \tag{8}$$
 
Numericamente, aplicando a mesma regra de quadratura discutida no Exercício 1, obtemos:
 
$$\langle \mu_k \rangle \approx \frac{1}{N} \sum_{n=1}^{N} \mu\!\left(T_k(s_n)\right) \tag{9}$$
 


##### Observando agora o efeito da hidráulica na placa térmica, podemos primeiro calcular a incluência da proximidade dos canos na condutividade térmica

In [ ]:
hydraulic_to_thermal = hydraulic_thermal.Hydraulic_to_Thermal(config)
hydraulic_to_thermal.run_problem1(plot= True, print_info=False)

##### Por fim, calculamos a rede térmica levando os microcanais em conta na hora de calcular a condutividade térmica

In [ ]:
hydraulic_to_thermal = hydraulic_thermal.Hydraulic_to_Thermal_P2(config)
hydraulic_to_thermal.run_problem2(plot=True)